# Đo chi phí suy luận trên Kaggle (mục 4.5 của báo cáo)

Notebook này làm đúng việc của **mục 12 trong `notebooks/colab_train.ipynb`**, nhưng trên Kaggle: chạy
`scripts/profile_cost.py` cho sáu mô hình trong **một phiên, trên một GPU**. Không huấn luyện gì. Khoảng 15 đến 25
phút trên T4.

## Trước khi chạy ô đầu tiên

1. **Dataset đã gắn vào notebook** (khung bên phải, *Input*), chứa các file sau, đặt ở đâu trong dataset cũng được:

   | File | Lấy ở đâu |
   |---|---|
   | `visfd_train.jsonl`, `visfd_dev.jsonl`, `visfd_test.jsonl` | `data/processed/` ở máy. **Cần cả ba**: BiLSTM dựng bảng từ vựng từ tập huấn luyện |
   | `bilstm_best.pt`, `lexicon_best.pt` | `checkpoints/<tên>/seed42/best.pt` ở máy, **đổi tên** như cột trái |
   | `asgcn_linked_best.pt`, `senticgcn_best.pt` | `nsmgat/checkpoints/<tên>/seed42/best.pt` trên Google Drive, tải về rồi **đổi tên** |
   | `phobert_best.pt`, `asgcn_best.pt` | chỉ khi bạn còn giữ; thiếu thì hai mô hình này được đo với trọng số lúc khởi tạo |

2. **Settings** (khung bên phải): *Accelerator* = **GPU T4 x2**; *Internet* = **On**. Internet cần để kéo mã từ GitHub
   và tải PhoBERT; Kaggle chỉ cho bật khi tài khoản đã xác minh số điện thoại.

Chạy lần lượt từ trên xuống. Ô nào dừng bằng dòng chữ in hoa thì làm theo dòng đó rồi chạy lại ô đó.

## 1. Kiểm GPU

In [ ]:
# Phai la T4: bon mo hinh dung PhoBERT duoc huan luyen tren Tesla T4 (Bang 4.6), va ca sau mo hinh phai do tren CUNG mot loai GPU.
import subprocess, torch

try:
    gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                         capture_output=True, text=True).stdout.strip()
except FileNotFoundError:
    gpu = ""
print(gpu or "(khong co nvidia-smi: notebook dang chay CPU)")
print("torch:", torch.__version__, "| CUDA:", torch.cuda.is_available())

assert torch.cuda.is_available(), "KHONG CO GPU. Settings > Accelerator > GPU T4 x2, doi may khoi dong lai, roi chay lai tu o nay."
TEN_GPU = torch.cuda.get_device_name(0)
assert "T4" in TEN_GPU, f"GPU DANG LA {TEN_GPU}, KHONG PHAI T4. Settings > Accelerator > GPU T4 x2, roi chay lai tu o nay."
print("OK:", TEN_GPU)

## 2. Lấy mã từ GitHub

In [ ]:
import subprocess
from pathlib import Path

# KHONG dat ten thu muc la "nsmgat": trung ten goi Python, import se nham sang thu muc nay.
REPO = Path("/kaggle/working/nsmgat-repo")
URL = "https://github.com/2611326-NguyenMinhTrong/nsmgat-vietnamese-absa.git"

if REPO.exists():
    subprocess.run(["git", "-C", str(REPO), "pull", "--ff-only"], check=True)
else:
    r = subprocess.run(["git", "clone", URL, str(REPO)], capture_output=True, text=True)
    assert r.returncode == 0, ("KHONG KEO DUOC MA. Bat Settings > Internet = On (can xac minh so dien thoai), roi chay lai o nay.\n"
                               + r.stderr[-500:])
# In commit dang chay, de sau nay biet so do sinh tu ma nguon nao
print(subprocess.run(["git", "-C", str(REPO), "log", "-1", "--oneline"], capture_output=True, text=True).stdout)

## 3. Cài mã nguồn

Không cài `requirements.txt`: file đó có `torch`, cài đè lên bản torch có CUDA sẵn của Kaggle là hỏng GPU.
Dòng `pip install -e` là bắt buộc vì mã nằm ở `src/nsmgat/`.

In [ ]:
%pip install -q transformers pyyaml
%pip install -q -e /kaggle/working/nsmgat-repo
# Kiem bang mot tien trinh moi: tien trinh cua notebook chua thay goi vua cai.
!python -c "import nsmgat, transformers; print('nsmgat:', nsmgat.__file__); print('transformers:', transformers.__version__)"

## 4. Nối dữ liệu và bộ trọng số từ dataset vào chỗ mã nguồn đọc

Ô này tìm file theo tên trong mọi dataset đã gắn, rồi tạo liên kết vào `data/processed/` và
`checkpoints/<tên>/seed42/best.pt` của mã nguồn. Ba file dữ liệu được kiểm số dòng và mã băm SHA-256, để chắc
đó đúng là dữ liệu đã cho các con số của báo cáo.

In [ ]:
import hashlib, os
from pathlib import Path

REPO = Path("/kaggle/working/nsmgat-repo")
GOC_VAO = Path("/kaggle/input")
MO_HINH = ("lexicon", "bilstm", "phobert", "asgcn", "asgcn_linked", "senticgcn")
SEED_DO = 42
# So dong va SHA-256 cua ba file o may hoc vien (do ngay 16/10/2026)
DU_LIEU = {
    "visfd_train.jsonl": (23872, "674db57d5a4969cb1cb1f0ea4757e94c9762d57a1e7d81488d4ab65971074790"),
    "visfd_dev.jsonl": (3316, "208653f1f02c423b0627036b2575716c8b3d6a859b10a69b11227c3204487060"),
    "visfd_test.jsonl": (6722, "f6d714f298b02476bb47adeb52ce6daf198d60f73cf8f7964e2253ebfea57a5e"),
}


def tim(ten):
    ds = sorted(p for p in GOC_VAO.rglob(ten) if p.is_file())
    assert len(ds) <= 1, f"CO {len(ds)} FILE TEN {ten}: {[str(p) for p in ds]}. Chi giu mot file trong dataset."
    return ds[0] if ds else None


def noi(nguon, dich):
    dich.parent.mkdir(parents=True, exist_ok=True)
    if dich.is_symlink() or dich.exists():
        dich.unlink()
    os.symlink(nguon, dich)


for ten, (so_dong, bam) in DU_LIEU.items():
    p = tim(ten)
    assert p is not None, f"THIEU {ten} TRONG DATASET. Can ca ba file train, dev, test (BiLSTM dung tap train de dung tu vung)."
    dem = sum(1 for _ in p.open(encoding="utf-8"))
    sha = hashlib.sha256(p.read_bytes()).hexdigest()
    assert dem == so_dong and sha == bam, f"{ten} KHAC FILE O MAY: {dem} dong (can {so_dong}), sha256 {sha[:16]} (can {bam[:16]})."
    noi(p, REPO / "data/processed" / ten)
    print(f"{ten:20} {dem:>7,} dong, khop file o may  <- {p}")

print()
co = []
for exp in MO_HINH:
    p = tim(f"{exp}_best.pt")
    if p is None:
        print(f"{exp:14} THIEU  -> se do voi trong so luc khoi tao")
        continue
    noi(p, REPO / "checkpoints" / exp / f"seed{SEED_DO}" / "best.pt")
    co.append(exp)
    print(f"{exp:14} CO {p.stat().st_size / 1e6:9.1f} MB  <- {p}")
print("\nCo best.pt:", ", ".join(co) or "(khong co)")

## 5. Đo

Chỉ cho thấy một GPU (`CUDA_VISIBLE_DEVICES=0`): T4 x2 có hai GPU, phép đo dùng một. Ô này in mỗi mô hình một dòng
và ghi `results/cost_profile.json`. Cột `khác` của `asgcn_linked` và `senticgcn` ở lô 32 phải bằng 0 hoặc rất nhỏ;
nếu lớn thì dừng và báo Claude.

In [ ]:
import os
os.chdir("/kaggle/working/nsmgat-repo")
!CUDA_VISIBLE_DEVICES=0 python scripts/profile_cost.py --seed {SEED_DO}

## 6. Kiểm kết quả và lấy file về

Ô này chép `cost_profile.json` ra `/kaggle/working/`, chạy lệnh kiểm `profile_cost.py --kiem` và in đường tải về.
Nếu dòng kết luận ghi **CHƯA dùng được** thì dừng và báo Claude trước khi đóng phiên.

In [ ]:
import os, shutil, subprocess
from pathlib import Path
from IPython.display import FileLink, display

REPO = Path("/kaggle/working/nsmgat-repo")
kq = REPO / "results/cost_profile.json"
assert kq.exists(), "CHUA CO results/cost_profile.json: o do o tren chua chay xong hoac da bao loi."
shutil.copy(kq, "/kaggle/working/cost_profile.json")

r = subprocess.run(["python", "scripts/profile_cost.py", "--kiem"], cwd=REPO, capture_output=True, text=True)
if r.returncode == 2 and "--kiem" in r.stderr:
    print("(Ma tren GitHub chua co lenh --kiem. Claude se kiem file nay o may.)")
else:
    print(r.stdout)

os.chdir("/kaggle/working")
display(FileLink("cost_profile.json"))  # bam vao duong nay de tai ve

## 7. Mang về máy

1. Tải `cost_profile.json`: bấm đường dẫn ở ô trên, hoặc mở khung bên phải *Output* (`/kaggle/working`), bấm vào
   dấu ba chấm cạnh `cost_profile.json` rồi chọn tải về. Đặt vào `results\cost_profile.json` của repo.
2. Lưu notebook còn nguyên output: *File > Download notebook*, đặt vào
   `notebooks\da-chay\cost\phien-<năm-tháng-ngày>-kaggle.ipynb`.
3. Tắt phiên để không tốn giờ GPU: nút nguồn ở góc trên, *Stop session*.
4. Báo Claude.

Không chép `checkpoints/` hay dataset về repo.